In [1]:
%load_ext autoreload
%autoreload 2
import eval
import os
import glob
import pickle

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)


In [7]:
out_dir = f"./out/onemotif_twostates_pos/"

motif = "3ixt"
motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
motif_out_dir = os.path.join(out_dir,motif)

In [9]:


def motif_results(motif_out_dir, motif_pdb):
    rows = []
    for design_dir in sorted(glob.glob(os.path.join(motif_out_dir, "design*"))):
        design_name = os.path.basename(design_dir)

        # load motif mask
        with open(os.path.join(design_dir, f"{motif}_spec.pkl"), "rb") as f:
            motif_mask = pickle.load(f)["motif_mask"]

        for state in [0, 1]:
            with open(os.path.join(design_dir, f"state{state}.pkl"), "rb") as f:
                outdict = pickle.load(f)

            for sample_idx in range(5):  # assume 5 samples per state
                pdb_file = os.path.join(design_dir, f"state{state}_sample{sample_idx}.pdb")
                if not os.path.exists(pdb_file):
                    continue

                rmsd = eval.motifRMSD(motif_pdb, pdb_file, motif_mask)

                rows.append({
                    "design": design_name,
                    "state": state,
                    "sample": sample_idx,
                    "motifrmsd": rmsd.item() if hasattr(rmsd, "item") else float(rmsd),
                    "plddt": outdict["plddt"].cpu().numpy()[sample_idx].mean(),
                    "ptm": outdict["ptm"].cpu().numpy()[sample_idx],
                })
                
    full = pd.DataFrame(rows)

    agg = full.groupby(["design", "state"]).agg(
        motifRMSD_mean=("motifrmsd", "mean"),
        motifRMSD_std=("motifrmsd", "std"),
        plddt=("plddt", "mean"),
        ptm=("ptm", "mean")
    ).reset_index()

    agg = agg.pivot(index="design", columns="state").reset_index()
    agg.columns = ["_".join(map(str, col)).rstrip("_") for col in agg.columns.to_flat_index()]
    agg = agg.rename(columns=lambda c: c.replace("_0", "_unbound").replace("_1", "_bound"))

    return full, agg



df_full,df_agg = motif_results(motif_out_dir,motif_pdb)
df_agg.head()

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
0,design0,2.052386,1.625973,0.231616,0.350459,0.526346,0.483885,0.249867,0.363347


In [32]:
# design_dir = f"./out/onemotif_twostates/{motif}/design1/"

# with open(os.path.join(design_dir, f"state1.pkl"),"rb") as f:
#     out = pickle.load(f)
# out

In [10]:


def plot_scatter_motifrmsd(df, motif):
    xrange = [0, 10]
    yrange = [0, 10]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("", "with std ± 1")
    )

    fig.add_trace(
        go.Scatter(
            x=df["motifRMSD_mean_unbound"],
            y=df["motifRMSD_mean_bound"],
            mode="markers",
            marker=dict(
                color=df["plddt_unbound"],
                colorscale="sunsetdark",
                showscale=True,
                colorbar=dict(title="pLDDT",outlinewidth=0)
            ),
            text=df["design"],
            name="Designs"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df["motifRMSD_mean_unbound"],
            y=df["motifRMSD_mean_bound"],
            mode="markers",
            marker=dict(
                color=df["plddt_unbound"],
                colorscale="sunsetdark",
                showscale=False
            ),
            text=df["design"],
            error_x=dict(array=df["motifRMSD_std_unbound"], color="gray", thickness=1),
            error_y=dict(array=df["motifRMSD_std_bound"], color="gray", thickness=1),
            name="Designs (err)"
        ),
        row=1, col=2
    )

    for c in [1, 2]:
        fig.add_shape(
            type="line", x0=0, y0=0, x1=10, y1=10,
            line=dict(color="lightgray", dash="dash"),
            row=1, col=c
        )

    fig.update_xaxes(range=xrange, title="Unbound motif RMSD", row=1, col=1)
    fig.update_yaxes(range=yrange, title="Bound motif RMSD", row=1, col=1)
    fig.update_xaxes(range=xrange, title="Unbound motif RMSD", row=1, col=2)
    fig.update_yaxes(range=yrange, title="Bound motif RMSD", row=1, col=2)

    fig.update_layout(
        width=1200, height=600,
        title=f"({motif}) Unbound vs Bound motif RMSD",
        yaxis_scaleanchor="x",
        yaxis2_scaleanchor="x2",
        showlegend=False
    )

    return fig


plot_scatter_motifrmsd(df_agg, motif)


### plot all

In [5]:
for motif in os.listdir(out_dir):
    motif_out_dir = os.path.join(out_dir,motif)
    motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
    full, agg = motif_results(motif_out_dir, motif_pdb)
    
    plot_scatter_motifrmsd(agg, motif).show()
